1. Isolation Forest 
Features are numeric and tabular (means, stds, proportions, deviations).

Dataset is reasonably large (~thousands of users), so Isolation Forest is scalable.

Handles imbalanced data naturally via contamination.

Provides direct anomaly scores, which is perfect for computing AUC.

In [ ]:

# 1. Imports

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from scipy.stats import entropy as sp_entropy


# 2. Feature Engineering Function

def engineer_features(interactions_df):
    df = interactions_df.copy()
    
    # --- Basic stats per user ---
    user_df = df.groupby("user")["rating"].agg(
        rating_mean="mean",
        rating_std="std",
        rating_median="median",
        rating_min="min",
        rating_max="max",
        rating_count="count"
    )
    user_df["rating_std"] = user_df["rating_std"].fillna(0)
    user_df["rating_range"] = user_df["rating_max"] - user_df["rating_min"]
    
    # --- Rating proportions & entropy ---
    rating_dist = df.groupby(["user","rating"]).size().unstack(fill_value=0)
    rating_dist = rating_dist.reindex(columns=range(6), fill_value=0)
    rating_props = rating_dist.div(rating_dist.sum(axis=1), axis=0)
    user_df["rating_entropy"] = rating_props.apply(lambda row: sp_entropy(row + 1e-9), axis=1)
    
    # Include proportions as features
    for i in range(6):
        user_df[f"prop_rating_{i}"] = rating_props[i]
    
    # --- Item popularity ---
    item_pop = df.groupby("item")["user"].count().rename("item_popularity")
    df_pop = df.merge(item_pop, left_on="item", right_index=True)
    pop_features = df_pop.groupby("user")["item_popularity"].agg(
        avg_item_popularity="mean",
        std_item_popularity="std"
    )
    pop_features["std_item_popularity"] = pop_features["std_item_popularity"].fillna(0)
    user_df = user_df.merge(pop_features, left_index=True, right_index=True, how="left")
    
    # --- Deviation from item average ---
    item_avg = df.groupby("item")["rating"].mean().rename("item_avg_rating")
    df_dev = df.merge(item_avg, left_on="item", right_index=True)
    df_dev["deviation"] = df_dev["rating"] - df_dev["item_avg_rating"]
    dev_features = df_dev.groupby("user")["deviation"].agg(
        mean_deviation="mean",
        std_deviation="std",
        abs_mean_deviation=lambda x: np.mean(np.abs(x))
    )
    dev_features["std_deviation"] = dev_features["std_deviation"].fillna(0)
    user_df = user_df.merge(dev_features, left_index=True, right_index=True, how="left")
    
    user_df = user_df.reset_index()
    return user_df


# 3. Load Training Data

data = np.load("training_batch_with_labels.npz")
X_train = pd.DataFrame(data["X"], columns=["user","item","rating"])
y_train = pd.DataFrame(data["y"], columns=["user","label"])

# Feature engineering
train_features = engineer_features(X_train)
train_df = train_features.merge(y_train, on="user")

feature_cols = [c for c in train_df.columns if c not in ["user","label"]]
X = train_df[feature_cols].values
y = train_df["label"].values

# Scale features (important for anomalies across features with different ranges)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# 4. Train Isolation Forest

iso = IsolationForest(n_estimators=57, contamination="auto", random_state=42) #56 works best so far, plateus at 1000+
iso.fit(X_scaled)

# Raw anomaly scores (higher = more anomalous)
anomaly_scores = -iso.decision_function(X_scaled)

# 5. Threshold tuning for F1

best_f1 = 0
best_threshold = 0
thresholds = np.linspace(np.min(anomaly_scores), np.max(anomaly_scores), 200)

for t in thresholds:
    y_pred = (anomaly_scores >= t).astype(int)
    f1 = f1_score(y, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

y_pred_final = (anomaly_scores >= best_threshold).astype(int)

# 6. Evaluation

auc = roc_auc_score(y, anomaly_scores)
precision = precision_score(y, y_pred_final)
recall = recall_score(y, y_pred_final)
f1 = f1_score(y, y_pred_final)

print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")

# 7. Save anomaly scores
np.savez("isolation_forest_submission.npz", predictions=anomaly_scores)
print("Anomaly scores saved to 'isolation_forest_submission.npz'")

Optimal Threshold: -0.0126
AUC: 0.6814
Precision: 0.2688
Recall: 0.5000
F1 Score: 0.3497
Anomaly scores saved to 'isolation_forest_submission.npz'


In [ ]:

import numpy as np
import pandas as pd
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
from scipy.stats import entropy as sp_entropy
from pyod.models.ecod import ECOD


# 2. Feature Engineering Function

def engineer_features(interactions_df):
    df = interactions_df.copy()
    
    # --- Basic stats per user ---
    user_df = df.groupby("user")["rating"].agg(
        rating_mean="mean",
        rating_std="std",
        rating_median="median",
        rating_min="min",
        rating_max="max",
        rating_count="count"
    )
    user_df["rating_std"] = user_df["rating_std"].fillna(0)
    user_df["rating_range"] = user_df["rating_max"] - user_df["rating_min"]
    
    # --- Rating proportions & entropy ---
    rating_dist = df.groupby(["user","rating"]).size().unstack(fill_value=0)
    rating_dist = rating_dist.reindex(columns=range(6), fill_value=0)
    rating_props = rating_dist.div(rating_dist.sum(axis=1), axis=0)
    user_df["rating_entropy"] = rating_props.apply(lambda row: sp_entropy(row + 1e-9), axis=1)
    
    # Include proportions as features
    for i in range(6):
        user_df[f"prop_rating_{i}"] = rating_props[i]
    
    # --- Item popularity ---
    item_pop = df.groupby("item")["user"].count().rename("item_popularity")
    df_pop = df.merge(item_pop, left_on="item", right_index=True)
    pop_features = df_pop.groupby("user")["item_popularity"].agg(
        avg_item_popularity="mean",
        std_item_popularity="std"
    )
    pop_features["std_item_popularity"] = pop_features["std_item_popularity"].fillna(0)
    user_df = user_df.merge(pop_features, left_index=True, right_index=True, how="left")
    
    # --- Deviation from item average ---
    item_avg = df.groupby("item")["rating"].mean().rename("item_avg_rating")
    df_dev = df.merge(item_avg, left_on="item", right_index=True)
    df_dev["deviation"] = df_dev["rating"] - df_dev["item_avg_rating"]
    dev_features = df_dev.groupby("user")["deviation"].agg(
        mean_deviation="mean",
        std_deviation="std",
        abs_mean_deviation=lambda x: np.mean(np.abs(x))
    )
    dev_features["std_deviation"] = dev_features["std_deviation"].fillna(0)
    user_df = user_df.merge(dev_features, left_index=True, right_index=True, how="left")
    
    user_df = user_df.reset_index()
    return user_df


# 3. Load Training Data

data = np.load("training_batch_with_labels.npz")
X_train = pd.DataFrame(data["X"], columns=["user","item","rating"])
y_train = pd.DataFrame(data["y"], columns=["user","label"])

# Feature engineering
train_features = engineer_features(X_train)
train_df = train_features.merge(y_train, on="user")

feature_cols = [c for c in train_df.columns if c not in ["user","label"]]
X = train_df[feature_cols].values
y = train_df["label"].values

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


# 4. Train Isolation Forest (with max_features tuning)

iso = IsolationForest(n_estimators=1000, contamination="auto", max_features=0.6, random_state=42)
iso.fit(X_scaled)

iso_scores = -iso.decision_function(X_scaled)


# 5. Train ECOD

ecod = ECOD()
ecod.fit(X_scaled)

ecod_scores = ecod.decision_scores_


# 6. Blend scores (normalize then combine)

iso_norm = MinMaxScaler().fit_transform(iso_scores.reshape(-1, 1)).ravel()
ecod_norm = MinMaxScaler().fit_transform(ecod_scores.reshape(-1, 1)).ravel()

blended_scores = 0.6 * iso_norm + 0.4 * ecod_norm


# 7. Threshold tuning for F1

best_f1 = 0
best_threshold = 0
thresholds = np.linspace(np.min(blended_scores), np.max(blended_scores), 200)

for t in thresholds:
    y_pred = (blended_scores >= t).astype(int)
    f1 = f1_score(y, y_pred)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = t

y_pred_final = (blended_scores >= best_threshold).astype(int)


# 8. Evaluation

auc = roc_auc_score(y, blended_scores)
precision = precision_score(y, y_pred_final)
recall = recall_score(y, y_pred_final)
f1 = f1_score(y, y_pred_final)

print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"AUC: {auc:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1 Score: {f1:.4f}")


# 9. Save anomaly scores

np.savez("isolation_forest_submission.npz", predictions=blended_scores)
print("Anomaly scores saved to 'isolation_forest_submission.npz'")

Optimal Threshold: 0.3719
AUC: 0.6562
Precision: 0.3471
Recall: 0.4200
F1 Score: 0.3801
Anomaly scores saved to 'isolation_forest_submission.npz'
